In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "westeurope"

apim_sku = 'Basicv2'

openai_resources = [ {"name": "openai1", "location": "uksouth"}]
openai_model_name = "gpt-4o-mini"
openai_model_version = "2024-07-18"
openai_model_sku = "GlobalStandard"
openai_deployment_name = "gpt-4o-mini"
openai_api_version = "2024-10-21"

build = 0
weather_mcp_server_image = "weather-mcp-server"
weather_mcp_server_src = "src/weather/mcp-server"

oncall_mcp_server_image = "oncall-mcp-server"
oncall_mcp_server_src = "src/oncall/mcp-server"

github_mcp_server_image = "github-mcp-server"
github_mcp_server_src = "src/github/mcp-server"

servicenow_mcp_server_image = "servicenow-mcp-server"
servicenow_mcp_server_src = "src/servicenow/mcp-server"
servicenow_instance_name = "" # Add here the name of your ServiceNow instance, e.g. "businessname-dev". Leave empty if you don't want to use ServiceNow.

utils.print_ok('Notebook initialized')

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM Gateway URL')
    apim_resource_name = utils.get_deployment_output(output, 'apimResourceName', 'APIM Resource Name')
    apim_subscription_key = utils.get_deployment_output(output, 'apimSubscriptionKey', 'APIM Subscription Key (masked)', True)
    app_insights_name = utils.get_deployment_output(output, 'applicationInsightsName', 'Application Insights Name')
    container_registry_name = utils.get_deployment_output(output, 'containerRegistryName', 'Container Registry Name')
    weather_containerapp_resource_name = utils.get_deployment_output(output, 'weatherMCPServerContainerAppResourceName', 'Weather Container App Resource Name')
    oncall_containerapp_resource_name = utils.get_deployment_output(output, 'oncallMCPServerContainerAppResourceName', 'Oncall Container App Resource Name')

    a2a_weather_containerapp_resource_name = utils.get_deployment_output(output, 'a2AWeatherAgentServerContainerAppResourceName', 'A2A (Weather) Agent Container App Resource Name')
    a2a_oncall_containerapp_resource_name = utils.get_deployment_output(output, 'a2AOncallAgentServerContainerAppResourceName', 'A2A (Oncall) Agent Container App Resource Name')

    a2a_weather_a2a_agent_ep = utils.get_deployment_output(output, 'a2AWeatherAgentServerContainerAppFQDN', 'A2A (Weather) Agent Endpoint')
    a2a_oncall_a2a_agent_ep = utils.get_deployment_output(output, 'a2AOncallAgentServerContainerAppFQDN', 'A2A (Oncall) Agent Endpoint')

inference_api_path = ""
inference_api_version = "2025-03-01-preview"

In [ ]:
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient


def build_chat_client() -> OpenAIChatCompletionClient:
    """Microsoft Agent Framework chat client routed through APIM."""
    return OpenAIChatCompletionClient(
        model=openai_model_name,
        azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key=apim_subscription_key,
        api_version=inference_api_version,
    )


async def ask_weather_agent(question: str) -> str:
    """Run a MAF agent that can call the remote Weather MCP tools."""
    async with MCPStreamableHTTPTool(
        name="Weather",
        url=f"{apim_resource_gateway_url}/weather/mcp",
        description="Remote Weather MCP tools via Streamable HTTP",
    ) as weather_tool:
        async with Agent(
            client=build_chat_client(),
            name="WeatherAgent",
            instructions=(
                "You are a helpful assistant. "
                "Use the 'Weather' tools when the user asks about weather or locations. "
                "Cite the source if appropriate."
            ),
        ) as agent:
            response = await agent.run(question, tools=weather_tool)
            return getattr(response, "text", None) or str(response)

In [ ]:
from typing import Literal
from mcp.server.fastmcp import FastMCP


def build_mcp_server() -> FastMCP:
    """Expose the MAF Weather agent as a Streamable-HTTP MCP server via FastMCP."""
    mcp = FastMCP(name="weather_maf", stateless_http=True, json_response=True)

    @mcp.tool(
        name="ask_weather",
        description="Ask the Weather agent a question; it may call the remote Weather MCP tools.",
    )
    async def ask(question: str) -> str:
        return await ask_weather_agent(question)

    return mcp


async def run(transport: Literal["sse", "stdio", "http"] = "stdio", port: int | None = None) -> None:
    mcp = build_mcp_server()

    if transport == "http" and port is not None:
        import uvicorn

        config = uvicorn.Config(mcp.streamable_http_app(), host="0.0.0.0", port=port, log_level="info")
        await uvicorn.Server(config).serve()
    elif transport == "sse" and port is not None:
        import uvicorn

        config = uvicorn.Config(mcp.sse_app(), host="0.0.0.0", port=port, log_level="info")
        await uvicorn.Server(config).serve()
    else:
        await mcp.run_stdio_async()

#### Blocking cell

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

## this a blocking call - it will block the running further cells
asyncio.run(run(transport="http", port=9090), debug=True)  # Change transport to "stdio" if you want to run it in stdio mode


#### Non-blocking cell

In [ ]:
import asyncio

# Schedule your long-running coroutine without blocking the cell
loop = asyncio.get_event_loop()             # Jupyter’s loop
mcp_agent_task = loop.create_task(run(transport="http", port=9090))
mcp_agent_task.set_name("mcp-agent")     # optional, helps with debugging

# (optional) surface exceptions instead of failing silently
def _report_done(t: asyncio.Task):
    try:
        t.result()
    except asyncio.CancelledError:
        print("mcp-agent: cancelled")
    except Exception as e:
        print("mcp-agent: crashed with:", repr(e))

mcp_agent_task.add_done_callback(_report_done)

print("Started in background:", mcp_agent_task)
# Now you can continue running other cells without waiting for the agent to finish


### Streamable HTTP Transport MCP Server Agent (Microsoft Agent Framework)


In [ ]:
import asyncio
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient

async def run_agent(url, prompt) -> None:
    # Connect to the local MCP server over streamable HTTP
    mcp_tool = MCPStreamableHTTPTool(
        name="weather",
        url=url,
        description="Weather MCP tools",
    )

    # Chat client routed through the APIM inference endpoint
    chat_client = OpenAIChatCompletionClient(
        model=openai_model_name,
        azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key=apim_subscription_key,
        api_version=inference_api_version,
    )

    async with mcp_tool:
        agent = Agent(
            client=chat_client,
            name="weather",
            instructions="You are a helpful assistant.",
        )
        async with agent:
            response = await agent.run(prompt, tools=[mcp_tool])
            print(response)

import nest_asyncio
nest_asyncio.apply()
asyncio.run(run_agent("http://127.0.0.1:9090/mcp", "What's the weather in Lisbon, Cairo and London?"))


### Stop the server

In [ ]:
# Ask it to stop
mcp_agent_task.cancel()

# Let it handle cancellation and swallow the CancelledError
await asyncio.gather(mcp_agent_task, return_exceptions=True)
